# Cleanup old processes

In [4]:
!pkill -u "$USER" -f '^clingcon '
!ps -aux | grep clingcon

unknown   274370 85.7  0.0  10352  3600 pts/2    Ss+  14:58   0:00 /bin/bash -c ps -aux | grep clingcon
unknown   274372  0.0  0.0   9692  2280 pts/2    S+   14:58   0:00 grep clingcon


# Run clingcon on instance with timeout, write result to result.out

In [8]:
import time, os

timeMax = "86400"
instanceName = "red_circuit.lp"
coreCount = "32"
use_tmux = True

outfile = "i_" + instanceName + "-t_" + timeMax + "-cpu_" + coreCount + "-" + time.strftime("%Y%m%d_%H%M%S") + ".log"
print(outfile)

instancePath = f"instances/{instanceName}"

if use_tmux:
    sessionName = "asp"
    !tmux new-session -d -s {sessionName} "bash -lc 'echo \"================ INSTANCE BEGIN ================\" > logs/{outfile}; echo \"File: {instancePath}\" >> logs/{outfile}; echo \"------------------------------------------------\" >> logs/{outfile}; cat {instancePath} >> logs/{outfile}; echo >> logs/{outfile}; echo \"================ CLINGCON OUTPUT ================\" >> logs/{outfile}; systemd-run --user --scope -p MemoryMax=128G -p MemoryHigh=120G -p MemorySwapMax=0 clingcon model2.lp game.lp {instancePath} --opt-mode=ignore -n 1 --quiet=1 -t {coreCount} --time {timeMax} >> logs/{outfile} 2>&1'"
    print(f"Started tmux session: {sessionName}")
    print(f"Attach with: tmux attach -d -t {sessionName}")
else:
    !bash -lc 'echo "================ INSTANCE BEGIN ================" > logs/{outfile}; echo "File: {instancePath}" >> logs/{outfile}; echo "------------------------------------------------" >> logs/{outfile}; cat {instancePath} >> logs/{outfile}; echo >> logs/{outfile}; echo "================ CLINGCON OUTPUT ================" >> logs/{outfile}; systemd-run --user --scope -p MemoryMax=128G -p MemoryHigh=120G -p MemorySwapMax=0 clingcon model2.lp game.lp {instancePath} --opt-mode=ignore -n 1 --quiet=1 -t {coreCount} --time {timeMax} >> logs/{outfile} 2>&1'

i_red_circuit.lp-t_86400-cpu_32-20260312_103954.log
Started tmux session: asp
Attach with: tmux attach -d -t asp


# Plot

In [1]:
from clingcon_plotly_viewer import ClingconAssignmentViewer

viewer = ClingconAssignmentViewer.from_log("logs/i_red_circuit.lp-t_86400-cpu_64-20260310_184714.log")
#viewer = ClingconAssignmentViewer.from_log("logs/" + outfile)
ui = viewer.interactive_explorer_widget(cols=4, show_zeros_3d=True)
display(ui)